In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/'Colab Notebooks'

/content/drive/MyDrive/Colab Notebooks


In [4]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [5]:
# MUST be first import after restart
import unsloth
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator

import torch
from torchvision import datasets, transforms
from datasets import Dataset

from trl import SFTTrainer, SFTConfig

# ----------------------------
# Load model
# ----------------------------
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/llava-1.5-7b-hf",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# LoRA (set dropout=0 for best Unsloth patching speed)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=16,
    lora_dropout=0,          # <--- important for Unsloth speed
    bias="none",
    target_modules="all-linear",
)

# ----------------------------
# MNIST -> vision "messages" dataset
# ----------------------------
tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
])

mnist = datasets.MNIST(root="./data", train=True, download=True, transform=tf)

instruction = "What digit is shown in this image? Answer with a single digit 0-9."

def to_conversation(i: int):
    img, label = mnist[i]
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text",  "text": instruction},
                    {"type": "image", "image": img},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": str(int(label))}
                ],
            },
        ]
    }

N = 2000
converted = [to_conversation(i) for i in range(N)]
train_ds = Dataset.from_list(converted)

# ----------------------------
# Train (use SFTTrainer + UnslothVisionDataCollator)
# ----------------------------
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=5,
        report_to="none",
        output_dir="./mnist_unsloth",
    ),
)

train_result = trainer.train()

print("train_runtime:", train_result.metrics.get("train_runtime"))

# ----------------------------
# Quick inference sanity check
# ----------------------------
FastVisionModel.for_inference(model)

img, gt = mnist[0]
messages = [
    {"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": instruction},
    ]}
]
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(img, prompt, add_special_tokens=False, return_tensors="pt").to("cuda")

out = model.generate(**inputs, max_new_tokens=2, temperature=0.0, use_cache=True)
print("GT:", int(gt))
print("Pred:", tokenizer.decode(out[0], skip_special_tokens=True))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Clip patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.04G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

...

Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients


100%|██████████| 9.91M/9.91M [00:02<00:00, 4.85MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.1MB/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 23,527,424 of 7,086,954,496 (0.33% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,1.973400
40,0.015400
60,0.005600
80,0.002600
100,0.004600
120,0.004000
140,0.002500
160,0.000700
180,0.004600
200,0.000700


train_runtime: 1467.7981
GT: 5
Pred: USER:  
What digit is shown in this image? Answer with a single digit 0-9. ASSISTANT: 5
